# API da busca de orientadores (Colab + GPU)

Roda o **pré-ranking** (busca híbrida no Qdrant) e o **reranking** (bge-reranker-v2-m3) na GPU do Colab
e expõe uma **API JSON com link público** para o seu front-end exibir o resultado.

**Antes de rodar:**
1. `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.
2. Na barra lateral, em 🔑 **Secrets**, crie (e libere o acesso deste notebook):
   - `QDRANT_URL` e `QDRANT_API_KEY` (os mesmos do `.env`);
   - `GITHUB_TOKEN` **só se o repositório for privado** (token com permissão de leitura).
3. Confira `BRANCH` na célula de clone: o Colab usa o código **que está no GitHub** (commit + push antes).
4. (Opcional) Ajuste `USAR_DRIVE` (cache do reranker no Google Drive) e `TEMAS_AQUECER` (temas do catálogo
   calculados já na inicialização).

Depois é só `Ambiente de execução > Executar tudo` (o Colab vai pedir permissão para acessar o Drive).
A célula do túnel imprime o link da API.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sem GPU: ative em Ambiente de execução > Alterar o tipo"

## 1. Código e dependências

In [ ]:
import os
from google.colab import userdata

REPO = "gavgms12/tcc_repository"
BRANCH = "buscas"
PASTA = "/content/tcc_repository"

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if os.path.exists(PASTA):
    !git -C {PASTA} fetch -q origin {BRANCH} && git -C {PASTA} checkout -q {BRANCH} && git -C {PASTA} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {url} {PASTA}
!git -C {PASTA} log -1 --oneline

In [ ]:
# O torch com CUDA já vem no Colab; aqui entram só as libs da busca e da API.
!pip install -q -r {PASTA}/requirements-busca.txt fastapi uvicorn pandas python-dotenv
# Se o pip pedir para reiniciar a sessão: reinicie e rode a partir da célula de clone (é rápido na 2ª vez).

## 2. Cache do reranker no Google Drive

As notas do reranker ficam num SQLite. Sem o Drive, ele começa vazio e some quando a sessão acaba
(a 1ª busca de cada tema roda o modelo; as seguintes saem do cache).
Com o Drive, o cache é copiado para a sessão no início e salvo de volta a cada 10 min (e na célula do fim):
cada tema/texto só é calculado uma vez **entre sessões**.

Para reaproveitar o cache do seu computador, envie `recuperacao/sistema_de_busca/cache/rerank_scores.sqlite`
para a pasta `CACHE_DRIVE` abaixo antes de rodar.

In [ ]:
import shutil, sqlite3

USAR_DRIVE = True                                   # False: cache só desta sessão
CACHE_DRIVE = "/content/drive/MyDrive/tcc_cache/rerank_scores.sqlite"
CACHE_LOCAL = "/content/rerank_scores.sqlite"       # o SQLite trabalha no disco local (mais rápido e seguro)

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(os.path.dirname(CACHE_DRIVE), exist_ok=True)
    if os.path.exists(CACHE_DRIVE) and not os.path.exists(CACHE_LOCAL):
        shutil.copy(CACHE_DRIVE, CACHE_LOCAL)

os.environ["RERANK_CACHE_PATH"] = CACHE_LOCAL       # lido pelo config.py
if os.path.exists(CACHE_LOCAL):
    with sqlite3.connect(CACHE_LOCAL) as con:
        print("Notas no cache:", con.execute("SELECT COUNT(*) FROM scores").fetchone()[0])
else:
    print("Cache vazio: as notas serão calculadas na 1ª busca de cada tema/texto.")


def salvar_cache_no_drive():
    """Cópia consistente do SQLite para o Drive (backup do sqlite, seguro mesmo com a API rodando)."""
    if not USAR_DRIVE or not os.path.exists(CACHE_LOCAL):
        return
    temporario = CACHE_DRIVE + ".tmp"
    with sqlite3.connect(CACHE_LOCAL) as origem, sqlite3.connect(temporario) as destino:
        origem.backup(destino)
    os.replace(temporario, CACHE_DRIVE)

## 3. Modelos e API

In [ ]:
import sys, torch

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.chdir(PASTA)
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)

print("GPU:", torch.cuda.is_available())

from recuperacao.sistema_de_busca.api import carregar_app
app = carregar_app()   # baixa/carrega bge-m3 e bge-reranker-v2-m3 (1ª vez demora alguns minutos)

In [ ]:
# Sobe a API em segundo plano (porta 8000) e aquece os modelos com uma busca de teste.
import threading, time, requests, uvicorn

servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
while not servidor.started:
    time.sleep(0.2)

r = requests.post("http://localhost:8000/recomendar", json={"texto": "algoritmos em grafos", "top_k": 3})
print(r.status_code, [p["nome"] for p in r.json()["ranking"]], r.json()["tempos"])

# salva o cache no Drive a cada 10 min enquanto a sessão estiver aberta
def _salvar_periodicamente():
    while True:
        time.sleep(600)
        try:
            salvar_cache_no_drive()
        except Exception as erro:
            print("Falha ao salvar o cache no Drive:", erro)
threading.Thread(target=_salvar_periodicamente, daemon=True).start()

### Temas do catálogo (opcional: aquecer o cache)

Os temas vêm de `TOPICOS` no `config.py` (lista em `GET /topicos`). A 1ª busca de cada tema roda o reranker
(~1.000 pares, segundos na GPU); depois sai do cache. Liste aqui os temas que vão ser usados (ex.: numa
apresentação) para já deixá-los calculados. `TEMAS_AQUECER = "todos"` aquece o catálogo inteiro.

In [ ]:
TEMAS_AQUECER = ["Sistemas Embarcados", "Internet das Coisas", "Inteligência Artificial"]

catalogo = requests.get("http://localhost:8000/topicos").json()["topicos"]
print(f"{len(catalogo)} temas no catálogo:", ", ".join(catalogo))

temas = catalogo if TEMAS_AQUECER == "todos" else TEMAS_AQUECER
desconhecidos = [t for t in temas if t not in catalogo]
if desconhecidos:
    print("⚠️ Fora do catálogo (confira o nome exato, com acentos):", desconhecidos)
for tema in [t for t in temas if t in catalogo]:
    r = requests.post("http://localhost:8000/recomendar", json={"topicos": [tema], "top_k": 5})
    print(f"  {tema:<40} {r.json()['tempos']['modelo_reranker']:6.1f} s no reranker")
salvar_cache_no_drive()

In [ ]:
# Link público (Cloudflare Quick Tunnel: sem conta). O link muda a cada vez que esta célula roda.
import re, subprocess

if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL_API = None
for linha in tunel.stdout:
    achado = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linha)
    if achado:
        URL_API = achado.group(0)
        break
# continua lendo o log em segundo plano (se o buffer encher, o túnel trava)
threading.Thread(target=lambda: [None for _ in tunel.stdout], daemon=True).start()
print("API:", URL_API)
print("Documentação interativa:", f"{URL_API}/docs")

## 4. Usando a API

`POST {URL_API}/recomendar` com JSON:

| campo | tipo | padrão | |
|---|---|---|---|
| `texto` | string | — | texto livre do aluno |
| `topicos` | lista | — | tópicos do catálogo (`GET /topicos`); pode combinar com `texto` |
| `top_k` | int | 5 | professores devolvidos |
| `n_itens` | int | 3 | evidências por tipo em `destaques` |
| `ano_min`, `ano_max` | int | — | filtro de ano dos itens |
| `incluir_pre_ranking` | bool | false | devolve também o ranking antes do reranking |

Resposta: `ranking` (lista de professores com `nome`, `score`, `score_inicial`, `scores_topicos`,
`perfil` = nota do texto de perfil e `parcela_do_score` (fração do score que veio dele),
`destaques` = melhores itens por tipo, e `topicos` = detalhe por tópico), `professores_cortados`, `tempos`
e, se pedido, `pre_ranking`.

Texto livre: frases de intenção ("quero pesquisar", "tenho interesse em"...) são removidas e o texto é
quebrado em ideias (`,` `;` `.` " e " " ou ") para o reranker; cada item fica com a maior nota entre elas.

Exemplo no front-end (JavaScript):
```js
const resp = await fetch(`${URL_API}/recomendar`, {
  method: "POST",
  headers: {"Content-Type": "application/json"},
  body: JSON.stringify({texto: "quero pesquisar sistemas embarcados e IoT", top_k: 5}),
  // ou um tema do catálogo:  {topicos: ["Sistemas Embarcados"], top_k: 5}
  // ou os dois juntos:       {topicos: ["Robótica"], texto: "drones para agricultura", top_k: 5}
});
const {ranking} = await resp.json();
```
Os temas válidos vêm de `GET {URL_API}/topicos` (dá para montar um menu de seleção no front-end com ela).

**Limites do Colab:** a sessão cai após ~90 min sem interação (ou ~12 h no total) e o link muda a cada execução.
Serve para testes e demonstração; para deixar no ar de forma fixa é preciso hospedar a API em um servidor.

In [ ]:
# Teste pelo link público (com fallback para localhost se o DNS ainda não propagou)
import json

def testar(pedido):
    try:
        r = requests.post(f"{URL_API}/recomendar", json=pedido)
    except requests.exceptions.ConnectionError:
        print("⚠️ O link público ainda está propagando no DNS. Testando localmente via localhost...")
        r = requests.post("http://localhost:8000/recomendar", json=pedido)
    r.raise_for_status()
    print("=" * 80, "\nPedido:", pedido)
    for prof in r.json()["ranking"]:
        perfil = prof["perfil"]
        parcela = f", {perfil['parcela_do_score']:.0%} do score" if perfil["parcela_do_score"] is not None else ""
        print(f"{prof['posicao']}º {prof['nome']}  (score {prof['score']:.4f})")
        print(f"    perfil: {perfil['score']:.3f}{parcela}")
        for tipo, itens in prof["destaques"].items():
            for item in itens:
                codigo = f"[{item['codigo']}] " if item.get("codigo") else ""
                print(f"    {tipo}: {codigo}{item['titulo']}  ({item['score']:.3f})")
    print("tempos:", json.dumps({k: round(v, 2) for k, v in r.json()["tempos"].items()}))

testar({"texto": "quero pesquisar sistemas embarcados e internet das coisas", "top_k": 5})   # texto livre
testar({"topicos": ["Sistemas Embarcados"], "top_k": 5})                                    # tema do catálogo
testar({"topicos": ["Robótica"], "texto": "drones para agricultura", "top_k": 5})            # tema + texto

## 5. Antes de encerrar

In [ ]:
# Salva no Drive as notas calculadas nesta sessão (também é feito sozinho a cada 10 min).
salvar_cache_no_drive()
with sqlite3.connect(CACHE_LOCAL) as con:
    print("Notas no cache:", con.execute("SELECT COUNT(*) FROM scores").fetchone()[0], "->", CACHE_DRIVE if USAR_DRIVE else "(só nesta sessão)")